# Tải các cohort GEO LUAD–LUSC và chia 5 fold theo bệnh nhân

Notebook này chuẩn bị các cohort đích trong review: `GSE37745`, `GSE30219`, `GSE43580`, `GSE10245`, và `GSE42127`. Mỗi cohort là **một dataset độc lập**; không gộp mẫu hay fit chuẩn hoá liên cohort ở đây.

Mỗi dataset xuất ra `expression.h5`, `metadata.csv`, `fold_assignment.csv`, `class_stats.csv`, `config.json`, và 15 file `folds/fold_<1..5>/{train,val,test}.h5`. Các file split có key `expression` (mẫu × gen) và `labels` (patient_id, class, label).

Việc chia dùng `StratifiedGroupKFold(groups=patient_id)`: mọi mẫu của một bệnh nhân chỉ thuộc **một** part P1–P5; tại một fold, tập bệnh nhân của train/val/test đôi một rời nhau. Notebook dừng bằng `assert` nếu vi phạm.

> Trước khi dùng để báo cáo, kiểm tra cột `unresolved_samples.csv`: notebook chỉ giữ mẫu có nhãn LUAD/LUSC rõ ràng từ metadata GEO và không tự suy đoán các loại mô học khác.

In [ ]:
import gzip, io, json, os, platform, re, shutil, sys, urllib.request
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

# Chỉnh hai đường dẫn này khi chạy trên Colab/Drive.
WORKDIR = Path(os.environ.get('KLTN_GEO_WORKDIR', 'D:/KLTN/geo_target_work')).resolve()
OUTDIR = Path(os.environ.get('KLTN_GEO_OUTDIR', 'D:/KLTN/experiment-data-geo-target')).resolve()
RAWDIR = WORKDIR / 'raw'
for d in (WORKDIR, RAWDIR, OUTDIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
SEED_SEARCH = range(SEED, SEED + 100)
N_FOLDS = 5
FOLD_PLAN = {r: {'test': r, 'val': r % N_FOLDS + 1,
                 'train': [p for p in range(1, N_FOLDS + 1) if p not in (r, r % N_FOLDS + 1)]}
             for r in range(1, N_FOLDS + 1)}

COHORTS = {
    # Các series dưới đây có một mẫu u cho mỗi người bệnh trong cohort LUAD/LUSC đã lọc.
    # GEO matrix không luôn công bố patient_id; fallback sample_id chỉ được bật rõ ràng ở các cohort này.
    'GSE37745': {'platform': 'GPL570', 'role': 'primary', 'one_sample_per_patient_verified': True},
    'GSE30219': {'platform': 'GPL570', 'role': 'primary', 'one_sample_per_patient_verified': True,
                 'luad_pattern': r'adenocarcinoma|\bluad\b|\badeno\b|adeno\s*carcinoma|\badk\d*\b'},
    # Matrix metadata has no patient identifier. The `clinical diagnosis patient:` field is a diagnosis, not an ID.
    # Leave fallback disabled until a verified sample-to-patient mapping is supplied.
    'GSE43580': {'platform': 'GPL570', 'role': 'primary', 'one_sample_per_patient_verified': False},
    'GSE10245': {'platform': 'GPL570', 'role': 'primary', 'one_sample_per_patient_verified': True},
    'GSE42127': {'platform': 'GPL6884', 'role': 'secondary_platform', 'one_sample_per_patient_verified': True},
}

print('raw:', RAWDIR)
print('out:', OUTDIR)
print('cohorts:', ', '.join(COHORTS))

## Test trước: ràng buộc bệnh nhân

Cell này là một kiểm tra nhỏ, có một bệnh nhân đóng góp hai mẫu. Nó phải chứng minh rằng hai mẫu này không bao giờ bị tách sang các vai trò khác nhau trong cùng một fold trước khi áp dụng cho dữ liệu GEO thật.

In [ ]:
def make_parts(meta, strat_col='label', n_splits=N_FOLDS, seeds=SEED_SEARCH):
    if meta['patient_id'].isna().any() or (meta['patient_id'].astype(str).str.strip() == '').any():
        raise ValueError('patient_id rỗng: không được chia fold trước khi sửa metadata')
    y = meta[strat_col].to_numpy()
    groups = meta['patient_id'].astype(str).to_numpy()
    if min(pd.Series(y).value_counts()) < n_splits:
        raise ValueError(f'Một lớp có ít hơn {n_splits} mẫu; không thể StratifiedGroupKFold 5 phần')
    ideal = pd.Series(y).value_counts().sort_index() / n_splits
    best = None
    for seed in seeds:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        part = np.empty(len(meta), dtype=int)
        for number, (_, held_out) in enumerate(splitter.split(np.zeros(len(meta)), y, groups), start=1):
            part[held_out] = number
        counts = pd.crosstab(part, y).reindex(columns=ideal.index, fill_value=0)
        score = float((counts - ideal.to_numpy()).abs().to_numpy().sum())
        if best is None or score < best[0]:
            best = (score, seed, part)
    return best[2], best[1], best[0]

def assign_fold_roles(meta):
    for fold, plan in FOLD_PLAN.items():
        roles = {part: 'train' for part in plan['train']}
        roles[plan['val']] = 'val'
        roles[plan['test']] = 'test'
        meta[f'fold_{fold}'] = meta['part'].map(roles)
    return meta

def verify_no_patient_leak(meta):
    assert (meta.groupby('patient_id')['part'].nunique() == 1).all(), 'Một bệnh nhân ở nhiều part'
    for fold in FOLD_PLAN:
        roles = meta[f'fold_{fold}']
        assert set(roles.unique()) == {'train', 'val', 'test'}
        patient_sets = {role: set(meta.loc[roles == role, 'patient_id'])
                        for role in ('train', 'val', 'test')}
        assert not patient_sets['train'] & patient_sets['val']
        assert not patient_sets['train'] & patient_sets['test']
        assert not patient_sets['val'] & patient_sets['test']
    roles = meta[[f'fold_{fold}' for fold in FOLD_PLAN]]
    assert ((roles == 'test').sum(axis=1) == 1).all()
    assert ((roles == 'val').sum(axis=1) == 1).all()

toy = pd.DataFrame({'patient_id': ['P1', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9'],
                    'label': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]})
toy['part'], _, _ = make_parts(toy)
toy = assign_fold_roles(toy)
verify_no_patient_leak(toy)
assert (toy.loc[toy.patient_id == 'P1', 'fold_1'].nunique() == 1)
print('PASS: bệnh nhân có nhiều mẫu không bị rò giữa train/val/test.')

## Tải và đọc GEO series matrix / annotation

Mã chỉ dùng tệp công khai của NCBI GEO. Nó lưu tệp thô để lần chạy sau không tải lại.

In [ ]:
def geo_stub(accession):
    prefix, number = accession[:3], accession[3:]
    return prefix + (number[:-3] if len(number) > 3 else '') + 'nnn'

def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or destination.stat().st_size == 0:
        print('download:', url)
        urllib.request.urlretrieve(url, destination)
    return destination

def matrix_urls(gse):
    # Uu tien cache: chay lai/debug local khong can goi mang neu matrix da tai xong.
    cached = sorted((RAWDIR / gse).glob('*series_matrix.txt.gz'))
    if cached:
        return cached
    base = f'https://ftp.ncbi.nlm.nih.gov/geo/series/{geo_stub(gse)}/{gse}/matrix/'
    with urllib.request.urlopen(base, timeout=120) as response:
        page = response.read().decode('utf-8', errors='replace')
    names = sorted(set(re.findall(r'href="([^"]*series_matrix\.txt\.gz)"', page)))
    if not names:
        raise RuntimeError(f'Không tìm thấy series matrix cho {gse}')
    return [base + name for name in names]

def parse_series_matrix(path):
    sample_rows, values, in_table = {}, [], False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as handle:
        for raw in handle:
            line = raw.rstrip('\r\n')
            if line.startswith('!series_matrix_table_begin'):
                in_table = True; continue
            if line.startswith('!series_matrix_table_end'):
                in_table = False; continue
            if in_table:
                values.append(line)
            elif line.startswith('!Sample_'):
                key, *row = line.split('\t')
                sample_rows.setdefault(key[1:], []).append([x.strip().strip('"') for x in row])
    flat = {}
    for key, rows in sample_rows.items():
        for occurrence, row in enumerate(rows, start=1):
            flat[key if len(rows) == 1 else f'{key}_{occurrence}'] = row
    metadata = pd.DataFrame(flat).set_index('Sample_geo_accession')
    metadata.index.name = 'sample_id'
    expression = pd.read_csv(io.StringIO('\n'.join(values)), sep='\t', index_col=0, low_memory=False)
    expression = expression.loc[:, expression.columns.intersection(metadata.index)]
    return expression.apply(pd.to_numeric, errors='coerce'), metadata

def load_gse(gse):
    matrices, metadata = [], []
    for source in matrix_urls(gse):
        path = source if isinstance(source, Path) else download(source, RAWDIR / gse / source.rsplit('/', 1)[-1])
        expr, pheno = parse_series_matrix(path)
        matrices.append(expr); metadata.append(pheno)
    X = pd.concat(matrices, axis=1)
    pheno = pd.concat(metadata, axis=0)
    X = X.loc[:, ~X.columns.duplicated()]
    pheno = pheno.loc[~pheno.index.duplicated(keep='first')]
    return X.loc[:, pheno.index], pheno

def load_platform_annotation(gpl):
    url = f'https://ftp.ncbi.nlm.nih.gov/geo/platforms/{geo_stub(gpl)}/{gpl}/annot/{gpl}.annot.gz'
    path = download(url, RAWDIR / gpl / f'{gpl}.annot.gz')
    rows, in_table = [], False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            if line.startswith('!platform_table_begin'):
                in_table = True; continue
            if line.startswith('!platform_table_end'):
                break
            if in_table:
                rows.append(line)
    annotation = pd.read_csv(io.StringIO(''.join(rows)), sep='\t', dtype=str)
    id_column = next((c for c in ('ID', 'ID_REF', 'Probe Set ID') if c in annotation), None)
    gene_column = next((c for c in ('Gene Symbol', 'Gene symbol', 'GENE_SYMBOL', 'Symbol', 'gene_assignment') if c in annotation), None)
    if id_column is None or gene_column is None:
        raise ValueError(f'{gpl}: không nhận diện được cột probe/gene; cột có: {list(annotation.columns)}')
    symbols = annotation.set_index(id_column)[gene_column].fillna('').str.split(' /// ').str[0].str.strip()
    return symbols[(symbols != '') & ~symbols.str.contains('///', regex=False)]

def collapse_probes(expression, symbols):
    symbols = symbols.reindex(expression.index).dropna()
    kept = expression.loc[symbols.index].dropna(axis=0, how='any')
    symbols = symbols.loc[kept.index]
    order = kept.var(axis=1).sort_values(ascending=False).index
    kept, symbols = kept.loc[order], symbols.loc[order]
    kept = kept.loc[~symbols.duplicated()]
    kept.index = pd.Index(symbols.loc[kept.index], name='gene')
    return kept.sort_index().T.astype(np.float32)

## Chuẩn hoá metadata và xác định nhãn

Nhãn chỉ được lấy khi metadata chứa dấu hiệu mô học rõ ràng. Các mẫu không rõ, hoặc khớp cả hai nhãn, được xuất riêng để kiểm tra thủ công thay vì bị gán nhãn suy đoán. `patient_id` ưu tiên trường patient/case/subject. Chỉ các cohort đã được khai báo `one_sample_per_patient_verified=True` mới được dùng `sample_id` thay thế; mọi cohort khác sẽ dừng để người chạy bổ sung ánh xạ bệnh nhân đã xác minh.

In [ ]:
# GEO GSE37745 ghi mô học là 'adeno' / 'squamous'; \b ngăn không bắt adenosquamous.
LUAD_PATTERN = r'adenocarcinoma|\bluad\b|\badeno\b|adeno\s*carcinoma'
LUSC_PATTERN = r'squamous|\blusc\b|squamous cell carcinoma|\bscc\b'
PATIENT_KEY_PATTERN = r'^(patient|case|subject|donor)([ _-]?(id|identifier|number|no\.?))?$'
REJECTED_PATIENT_KEY_PATTERN = r'diagnosis|histolog|disease|tumou?r|cancer|specimen|tissue'

def characteristics_text(metadata):
    cols = [c for c in metadata.columns if c.startswith('Sample_characteristics') or c in ('Sample_title', 'Sample_source_name_ch1')]
    return metadata[cols].fillna('').astype(str).agg(' | '.join, axis=1)

def extract_patient_id(metadata, text, allow_sample_id_fallback=False):
    # A generic `characteristics` column may say `clinical diagnosis patient:`.
    # Its value is a diagnosis, never an identifier; require an explicit ID-like field name.
    patient_columns = [c for c in metadata.columns
                       if re.fullmatch(PATIENT_KEY_PATTERN, c.replace('Sample_', '').replace('_ch1', '').replace('_', ' ').strip(), flags=re.I)]
    for column in patient_columns:
        candidate = metadata[column].astype(str).str.replace(r'^.*?:', '', regex=True).str.strip()
        if (candidate != '').all() and candidate.nunique() > 1:
            return candidate, f'metadata:{column}'
    characteristic_columns = [c for c in metadata.columns if c.startswith('Sample_characteristics')]
    for column in characteristic_columns:
        raw = metadata[column].fillna('').astype(str).str.strip().str.strip('\"')
        key = raw.str.extract(r'^([^:]+):', expand=False).str.strip().str.lower()
        if key.notna().all() and key.nunique() == 1:
            field = key.iloc[0]
            if (re.fullmatch(PATIENT_KEY_PATTERN, field, flags=re.I) and
                    not re.search(REJECTED_PATIENT_KEY_PATTERN, field, flags=re.I)):
                candidate = raw.str.replace(r'^[^:]+:', '', regex=True).str.strip()
                if (candidate != '').all() and candidate.nunique() > 1:
                    return candidate, f'characteristics:{field}'
    if allow_sample_id_fallback:
        return pd.Series(metadata.index, index=metadata.index), 'verified one-sample-per-patient: sample_id'
    raise ValueError('Không tìm thấy patient_id cho mọi mẫu. Dừng để bổ sung ánh xạ bệnh nhân đã xác minh; không được dùng sample_id thay thế.')

def label_luad_lusc(metadata, allow_sample_id_fallback=False, luad_pattern=LUAD_PATTERN, lusc_pattern=LUSC_PATTERN):
    text = characteristics_text(metadata)
    luad = text.str.contains(luad_pattern, case=False, regex=True)
    lusc = text.str.contains(lusc_pattern, case=False, regex=True)
    mixed = text.str.contains(r'adenosquamous', case=False, regex=True)
    luad, lusc = luad & ~mixed, lusc & ~mixed
    label = pd.Series(pd.NA, index=metadata.index, dtype='Int64')
    label.loc[luad & ~lusc] = 0
    label.loc[lusc & ~luad] = 1
    patient_id, source = extract_patient_id(metadata, text, allow_sample_id_fallback)
    out = pd.DataFrame({'patient_id': patient_id.astype(str), 'label': label,
                        'class': label.map({0: 'LUAD', 1: 'LUSC'}),
                        'metadata_text': text, 'luad_match': luad, 'lusc_match': lusc}, index=metadata.index)
    out.index.name = 'sample_id'
    return out, source

def class_stats(meta):
    rows = []
    views = [('all', 'all', pd.Series(True, index=meta.index))]
    views += [('part', f'P{part}', meta.part == part) for part in range(1, N_FOLDS + 1)]
    views += [(f'fold_{fold}', role, meta[f'fold_{fold}'] == role)
              for fold in FOLD_PLAN for role in ('train', 'val', 'test')]
    for fold, split, mask in views:
        subset = meta.loc[mask]
        for label, name in ((0, 'LUAD'), (1, 'LUSC'), (None, 'TOTAL')):
            values = subset if label is None else subset.loc[subset.label == label]
            rows.append({'fold': fold, 'split': split, 'class': name,
                         'n_samples': len(values), 'n_patients': values.patient_id.nunique()})
    return pd.DataFrame(rows)

## Dựng dataset và xuất 5 × train/val/test HDF5

`expression.h5` là ma trận đầy đủ đã lọc. Mỗi file fold chỉ chứa các mẫu của đúng vai trò đó; labels được lưu tách trong key `labels`, không được chèn vào ma trận gen.

In [ ]:
def write_h5_split(path, X, meta):
    path.parent.mkdir(parents=True, exist_ok=True)
    X.to_hdf(path, key='expression', mode='w', complevel=4, complib='zlib')
    meta[['patient_id', 'class', 'label']].to_hdf(path, key='labels', mode='a', complevel=4, complib='zlib')

def build_cohort(gse, spec):
    raw_expression, raw_meta = load_gse(gse)
    if not raw_meta['Sample_platform_id'].eq(spec['platform']).all():
        raise ValueError(f"{gse}: platform thực tế {raw_meta['Sample_platform_id'].unique().tolist()} không phải {spec['platform']}")
    labels, patient_source = label_luad_lusc(raw_meta, spec.get('one_sample_per_patient_verified', False),
                                               spec.get('luad_pattern', LUAD_PATTERN),
                                               spec.get('lusc_pattern', LUSC_PATTERN))
    unresolved = raw_meta.join(labels).loc[labels.label.isna()].copy()
    keep = labels.label.notna()
    meta = raw_meta.loc[keep].join(labels.loc[keep, ['patient_id', 'class', 'label']])
    meta['label'] = meta['label'].astype(int)
    if set(meta['class']) != {'LUAD', 'LUSC'}:
        raise ValueError(f"{gse}: sau lọc không đủ hai lớp: {meta['class'].value_counts().to_dict()}")
    symbols = load_platform_annotation(spec['platform'])
    expression = raw_expression.loc[:, meta.index]
    X = collapse_probes(expression, symbols).loc[meta.index]
    meta = meta.loc[X.index].copy()
    meta['part'], selected_seed, imbalance = make_parts(meta)
    meta = assign_fold_roles(meta)
    verify_no_patient_leak(meta)

    dataset_dir = OUTDIR / gse.lower()
    if dataset_dir.exists():
        raise FileExistsError(f'{dataset_dir} already exists. Inspect or move it before regenerating; never overwrite split artifacts automatically.')
    dataset_dir.mkdir(parents=True)
    X.to_hdf(dataset_dir / 'expression.h5', key='expression', mode='w', complevel=4, complib='zlib')
    meta.to_csv(dataset_dir / 'metadata.csv', index_label='sample_id')
    assignment = meta[['patient_id', 'class', 'label', 'part'] + [f'fold_{f}' for f in FOLD_PLAN]]
    assignment.to_csv(dataset_dir / 'fold_assignment.csv', index_label='sample_id')
    stats = class_stats(meta)
    stats.to_csv(dataset_dir / 'class_stats.csv', index=False)
    unresolved.to_csv(dataset_dir / 'unresolved_samples.csv', index_label='sample_id')
    for fold in FOLD_PLAN:
        for role in ('train', 'val', 'test'):
            subset = meta.index[meta[f'fold_{fold}'] == role]
            write_h5_split(dataset_dir / 'folds' / f'fold_{fold}' / f'{role}.h5', X.loc[subset], meta.loc[subset])
    config = {'dataset': gse, 'task': 'LUAD vs LUSC on one GEO target cohort',
              'platform': spec['platform'], 'role': spec['role'],
              'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
              'source': f'NCBI GEO {gse} series matrix', 'n_samples': len(meta),
              'n_patients': int(meta.patient_id.nunique()), 'n_genes': int(X.shape[1]),
              'label_mapping': {'LUAD': 0, 'LUSC': 1}, 'patient_id_source': patient_source,
              'one_sample_per_patient_verified': spec.get('one_sample_per_patient_verified', False),
              'split_method': 'StratifiedGroupKFold with patient_id groups; fold r: test=P_r, val=P_(r+1), train=remaining parts',
              'seed': SEED, 'selected_seed': int(selected_seed), 'split_imbalance_score': imbalance,
              'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                           'pandas': pd.__version__, 'scikit-learn': sklearn.__version__},
              'files': ['expression.h5', 'metadata.csv', 'fold_assignment.csv', 'class_stats.csv',
                        'unresolved_samples.csv', 'folds/fold_<1..5>/{train,val,test}.h5']}
    (dataset_dir / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{gse}: {len(meta)} samples / {meta.patient_id.nunique()} patients / {X.shape[1]} genes; seed={selected_seed}')
    print(meta['class'].value_counts().to_dict(), '| unresolved:', len(unresolved))
    return dataset_dir

built = [build_cohort(gse, spec) for gse, spec in COHORTS.items()]

## Kiểm tra sau khi ghi file

Đọc lại từ đĩa, không dựa vào biến trong bộ nhớ. Điều kiện pass là index/nhãn khớp, 15 file split tồn tại, và không có giao bệnh nhân ở bất kỳ fold nào.

In [ ]:
for dataset_dir in built:
    X = pd.read_hdf(dataset_dir / 'expression.h5', key='expression')
    assignment = pd.read_csv(dataset_dir / 'fold_assignment.csv', index_col='sample_id')
    assert list(X.index) == list(assignment.index)
    verify_no_patient_leak(assignment)
    for fold in FOLD_PLAN:
        expected_ids = set()
        for role in ('train', 'val', 'test'):
            path = dataset_dir / 'folds' / f'fold_{fold}' / f'{role}.h5'
            split_X = pd.read_hdf(path, key='expression')
            split_y = pd.read_hdf(path, key='labels')
            expected = assignment.index[assignment[f'fold_{fold}'] == role]
            assert list(split_X.index) == list(expected) == list(split_y.index)
            assert set(split_y['patient_id']) == set(assignment.loc[expected, 'patient_id'])
            expected_ids.update(split_X.index)
        assert expected_ids == set(X.index)
    print('PASS:', dataset_dir.name, '| 5 folds × train/val/test H5, no patient leakage')